# ATLAS — Deep Learning (Colab GPU)

Bu notebook ATLAS'ning butun deep learning quvurini GPU'da ishga tushiradi va natijani saytga qaytaradi.

| Bosqich | Nima qilinadi | Model |
|---|---|---|
| 1. Ma'lumot | 37 aktiv × 15 yil narx, FNSPID (2012–2023) + GDELT + o'z kollektorimiz yangiliklari | — |
| 2. Matn | Har bir sarlavha → FinBERT (muzlatilgan, fp16) → 768 → PCA 64 | Transformer (BERT) |
| 3. Oldindan o'qitish | Narx ketma-ketliklarida **masked reconstruction** (self-supervised) | Transformer encoder |
| 4. Qidiruv | Optuna TPE + pruning: o'lcham, chuqurlik, oyna, dropout, lr | — |
| 5. Asosiy model | **AtlasNet**: narx Transformeri + bozorlararo attention + yangilik attention + mavsumiylik → gated fusion → kvantillar (q05/q50/q95 × 1/5/20 kun) | ko'p shoxli Transformer, ansambl |
| 6. Baholash | Walk-forward (5 fold) + ablation (yangiliksiz, oldindan o'qitishsiz...) + Diebold–Mariano + Holm | — |
| 7. Holdout | Oxirgi 1 yil — **faqat bir marta** | — |
| 8. Signal modellari | SetupNet (SNR/SND/klassik figuralar, 82k signal) va oltin 5m–4h | 1D-CNN + attention |
| 9. Eksport | `data` shoxchasiga yoki zip | — |

**Halollik qoidalari** (loyihaning o'zagi): yangilik faqat paydo bo'lgan vaqtidan keyin ishlatiladi; bozor konteksti 1 kun lag; PCA va oldindan o'qitish faqat eng birinchi test davridan oldingi ma'lumotda; holdoutga qarab hech narsa tanlanmaydi.

> **Runtime → Change runtime type → GPU** (T4 yetadi; A100 bo'lsa `PRESET = "gpu_xl"`).

### 💾 Natijalar saqlanadi
Hamma narsa Google Drive'dagi `atlas_data` papkasiga yoziladi. Keyingi safar **Run all** bosilganda o'qitish qayta bajarilmaydi —
saqlangan model va hisobotlar 1–2 daqiqada ko'rsatiladi. Qaytadan o'qitish kerak bo'lsa: `RETRAIN = True`.

### ⚡ Tezkor rejim (standart)
`QUICK = True` — **~20–30 daqiqa**: tayyor narx va yangilik ma'lumoti (`data` shoxchasidan) olinadi, yuklab olish va FinBERT qayta hisoblanmaydi;
oldindan o'qitish 5 epoch, `fast` model (2 fold walk-forward, 2 modelli ansambl); SetupNet va oltin modellari qayta o'qitilmaydi — tayyor hisobotlari ko'rsatiladi.
To'liq, kuchli versiya uchun `QUICK = False`, `PRESET = "gpu"` (bir necha soat).

In [ ]:
#@title 0. Sozlamalar
QUICK = True            #@param {type:"boolean"}
RETRAIN = False         #@param {type:"boolean"}
# RETRAIN=False: Drive'da oldingi natija bo'lsa, qayta o'qitilmaydi — faqat natijalar ko'rsatiladi (1–2 daqiqa)
PRESET = "gpu"          #@param ["gpu", "gpu_xl", "fast", "default", "cpu"]
USE_DRIVE = True        #@param {type:"boolean"}
USE_FNSPID = True       #@param {type:"boolean"}
FNSPID_PER_DAY = 80     #@param {type:"integer"}
GDELT_DAYS = 30         #@param {type:"integer"}
PRETRAIN_EPOCHS = 30    #@param {type:"integer"}
TUNE_TRIALS = 20        #@param {type:"integer"}
RUN_HOLDOUT = False     #@param {type:"boolean"}
REPO = "https://github.com/ilhomjonshoyimov10-cyber/atlas"
if QUICK:
    PRESET, GDELT_DAYS, PRETRAIN_EPOCHS, TUNE_TRIALS, USE_FNSPID = "fast", 0, 5, 0, False
print("Rejim:", "TEZKOR" if QUICK else "TO'LIQ", "| preset:", PRESET)

In [ ]:
#@title 1. GPU va muhit
import os, subprocess, json, time
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "GPU topilmadi — Runtime > Change runtime type > GPU")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/atlas_data"     # sessiya uzilsa ham ma'lumot va modellar saqlanadi
else:
    DATA = "/content/atlas_data"
os.makedirs(DATA, exist_ok=True)
os.environ["ATLAS_DATA"] = DATA
os.environ["ATLAS_FNSPID_PER_DAY"] = str(FNSPID_PER_DAY)
print("ATLAS_DATA =", DATA)

In [ ]:
#@title 2. Kod va kutubxonalar
%cd /content
if not os.path.exists("/content/atlas"):
    !git clone -q {REPO} /content/atlas
else:
    !cd /content/atlas && git pull -q
%cd /content/atlas
!pip -q install -r requirements.txt optuna huggingface_hub
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
#@title 3. Mavjud ma'lumotni olish (sayt ishlatayotgan `data` shoxchasi: kollektor yangiliklari, narxlar, jurnallar)
if not os.path.exists(f"{DATA}/news.sqlite"):
    !git clone -q --depth 1 --branch data {REPO} /content/atlas_state
    !cp -rn /content/atlas_state/. {DATA}/ && rm -rf {DATA}/.git
!ls {DATA}

## 1. Narxlar — 15 yil, 37 aktiv

In [ ]:
if QUICK:
    print("Tezkor rejim: narxlar data shoxchasidan olindi (yuklab olinmaydi)")
else:
    !python -m atlas.cli download --years 15

## 2. Yangiliklar
* **FNSPID** — Nasdaq moliyaviy yangiliklari (2012–2023, vaqt belgisi bilan). Fayl katta (bir necha GB) — yuklab olish 10–20 daqiqa.
  Loyihadagi aksiyalar haqidagi sarlavhalar to'liq olinadi, qolganidan kuniga `FNSPID_PER_DAY` ta namuna.
* **GDELT** — so'nggi kunlar (DOC API). Server tez-tez `429 Too Many Requests` qaytaradi — bu xato emas, kod kutib qayta urinadi; 30 kun ~5–10 daqiqa.
* **O'z kollektorimiz** — `news.sqlite` (birinchi ko'rilgan vaqti bilan).

Yangilik modelga faqat `available_at` (paydo bo'lgan vaqt) bo'yicha kiradi: soat 20:00 UTC dan keyin chiqqan xabar ertangi qarorga o'tadi.

In [ ]:
os.makedirs(f"{DATA}/news/raw", exist_ok=True)
if QUICK: print("Tezkor rejim: yangiliklar data shoxchasidagi kollektordan olinadi")
if USE_FNSPID and not os.path.exists(f"{DATA}/news/raw/nasdaq_exteral_data.csv"):
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download("Zihan1004/FNSPID", "Stock_news/nasdaq_exteral_data.csv", repo_type="dataset",
                            local_dir="/content/fnspid")
        !ln -sf {p} {DATA}/news/raw/nasdaq_exteral_data.csv
        print("FNSPID tayyor:", p)
    except Exception as e:
        print("FNSPID yuklanmadi (HF sahifasida shartlarni qabul qilish kerak bo'lishi mumkin):", e)
if GDELT_DAYS:
    !python -m atlas.cli gdelt --days {GDELT_DAYS}

In [ ]:
#@title Yangiliklar soni (yil bo'yicha)
import pandas as pd, matplotlib.pyplot as plt
from atlas.data.news_open import load_all_news
news = load_all_news()
print(len(news), "sarlavha |", news.origin.value_counts().to_dict())
news.groupby([news.available_at.dt.year, "origin"]).size().unstack(fill_value=0).plot.bar(stacked=True, figsize=(10, 3), title="Yangiliklar soni")
plt.show()
del news

## 3. FinBERT embedding → kunlik yangilik kubi
FinBERT muzlatilgan (o'qitilmaydi): har sarlavhadan 768 o'lchamli vektor va ton (ijobiy − salbiy) olinadi, PCA bilan 64 ga tushiriladi.
PCA **faqat holdoutdan oldingi** yangiliklarda fit qilinadi. Kub: kun × (umumiy / guruh / aktiv) × [o'rtacha embedding, soni, ton, voqea turi].

In [ ]:
t0 = time.time()
if QUICK and os.path.exists(f"{DATA}/embeddings/news_cube.npz"):
    print("Tezkor rejim: tayyor FinBERT kubi ishlatiladi")
else:
    !python -m atlas.cli features --embedder finbert
print(f"{(time.time()-t0)/60:.1f} daqiqa")

## 4. Self-supervised oldindan o'qitish (masked reconstruction)
Narx Transformeri 37 aktivning ko'p yillik ketma-ketliklarida belgisiz vazifada o'qitiladi: oynadagi kunlarning ~30%i bloklar bilan yashiriladi va model ularni tiklaydi.
Faqat **eng birinchi walk-forward test davridan oldingi** ma'lumot ishlatiladi. Keyin AtlasNet shu og'irliklardan boshlanadi, foydasi esa `full` va `full_scratch` variantlarini solishtirib o'lchanadi.

In [ ]:
if RETRAIN or not os.path.exists(f"{DATA}/reports/pretrain.json"):
    !python -m atlas.cli pretrain --preset {PRESET} --device cuda --pt-epochs {PRETRAIN_EPOCHS}
else:
    print("Saqlangan natija ishlatildi (qayta o'qitish uchun RETRAIN = True)")
r = json.load(open(f"{DATA}/reports/pretrain.json"))
h = pd.DataFrame(r["history"]).set_index("epoch")
ax = h.plot(figsize=(8, 3), title=f"Masked reconstruction MSE (o'rtacha bilan to'ldirish tayanchi: {r['baseline_mean_fill_mse']:.3f})")
ax.axhline(r["baseline_mean_fill_mse"], ls="--", c="gray"); plt.show()

## 5. Giperparametr qidiruvi (Optuna)
TPE sampler + MedianPruner. Qidiruv faqat walk-forward'ning **birinchi** foldlarida baholanadi; oxirgi fold tekshiruv uchun yashiriladi, holdoutga umuman tegilmaydi.
Agar tanlangan sozlama oxirgi foldda yaxshilanish bermasa — notebook avtomatik ravishda preset sozlamasida qoladi.

In [ ]:
USE_TUNED = False
if TUNE_TRIALS and (RETRAIN or not os.path.exists(f"{DATA}/reports/tuning.json")):
    !python -m atlas.cli tune --preset {PRESET} --device cuda --trials {TUNE_TRIALS} --epochs 15
    tr = json.load(open(f"{DATA}/reports/tuning.json"))
    print(tr["check_verdict"])
    USE_TUNED = tr["check_fold_loss_best"] < tr["check_fold_loss_base"]
    hist = pd.DataFrame([t for t in tr["history"] if t["value"] is not None])
    hist.plot(x="n", y="value", style="o-", figsize=(8, 3), title="Optuna: har bir urinish (kichik = yaxshi)"); plt.show()
TUNED_FLAG = "--tuned" if USE_TUNED else ""
print("Tanlangan sozlama ishlatiladimi:", USE_TUNED)

## 6. Walk-forward baholash + ablation
Har bir fold: o'qitish → validatsiya (erta to'xtatish, konformal kalibratsiya) → 6 oylik test. Variantlar:

| variant | narx | mavsum | bozorlararo | yangilik | oldindan o'qitish |
|---|---|---|---|---|---|
| climatology | statistik tayanch | | | | |
| price | ✓ | | | | ✓ |
| price+season | ✓ | ✓ | | | ✓ |
| no_news | ✓ | ✓ | ✓ | | ✓ |
| **full** | ✓ | ✓ | ✓ | ✓ | ✓ |
| full_scratch | ✓ | ✓ | ✓ | ✓ | — |

Solishtirish: kunlik pinball farqi bo'yicha Diebold–Mariano (Newey–West) testi, ko'p taqqoslash uchun Holm tuzatishi.

In [ ]:
t0 = time.time()
if RETRAIN or not os.path.exists(f"{DATA}/reports/report_walkforward.json"):
    !python -m atlas.cli evaluate --preset {PRESET} --device cuda {TUNED_FLAG} --pretrained {DATA}/models/pretrained.pt
else:
    print("Saqlangan natija ishlatildi (qayta o'qitish uchun RETRAIN = True)")
print(f"{(time.time()-t0)/60:.1f} daqiqa")
rep = json.load(open(f"{DATA}/reports/report_walkforward.json"))
display(pd.DataFrame(rep["walk_forward_summary"]).T.sort_values("pinball").style.format("{:.4f}").set_caption("Walk-forward (kichik pinball = yaxshi)"))
display(pd.DataFrame(rep["ablation_tests"]).T)

In [ ]:
#@title Fold bo'yicha natijalar
rows = []
for f in rep["folds"]:
    for v, m in f["metrics"].items():
        rows.append({"fold": f["fold"], "variant": v, "pinball": m["pinball"]})
pv = pd.DataFrame(rows).pivot(index="fold", columns="variant", values="pinball")
(pv.sub(pv["climatology"], axis=0) * -1).plot.bar(figsize=(10, 3), title="Climatology'dan qanchaga yaxshi (katta = yaxshi)"); plt.show()

## 7. Final holdout — faqat BIR marta
Hamma qarorlar (arxitektura, preset, qidiruv, oldindan o'qitish) qabul qilingandan keyin. Holdout natijasiga qarab orqaga qaytib sozlama o'zgartirilsa, u endi "tegilmagan" bo'lmaydi — tizim buni `experiments.jsonl` da qayd qiladi.

In [ ]:
if RUN_HOLDOUT:
    !python -m atlas.cli evaluate --final --preset {PRESET} --device cuda {TUNED_FLAG} --pretrained {DATA}/models/pretrained.pt
    fin = json.load(open(f"{DATA}/reports/report_final.json"))
    ho = [f for f in fin["folds"] if f["fold"] == "HOLDOUT"][0]
    display(pd.DataFrame({v: {"pinball": m["pinball"], **{f"cov90_h{h}": m[f"h{h}"]["coverage90"] for h in (1, 5, 20)},
                              **{f"dir_h{h}": m[f"h{h}"]["dir_acc"] for h in (1, 5, 20)}} for v, m in ho["metrics"].items()}).T)
    !python -m atlas.cli backtest --threshold 0.05 --cost-bps 5
else:
    print("RUN_HOLDOUT=False — holdout hali tegilmagan.")

## 8. Servis modeli (saytda ishlaydigan ansambl)
Barcha ma'lumotda (oxirgi 10% — erta to'xtatish va konformal kalibratsiya) `PRESET` bo'yicha ansambl o'qitiladi.

In [ ]:
if QUICK:
    print("Tezkor rejim: servis modeli qayta o'qitilmaydi (saytdagisi ishlaydi)")
else:
    !python -m atlas.cli train --preset {PRESET} --device cuda {TUNED_FLAG} --pretrained {DATA}/models/pretrained.pt

## 9. Strategiya signallari: SetupNet va oltin 5m–4h
* **SetupNet** — SNR, SND, BOS, klassik figuralar, RSI/MA, shamlar bo'yicha 37 aktivdan ~82 ming signal; 1D-CNN (dilated) + attention pooling + embeddinglar.
* **Oltin ichki kun** — 5m/15m/30m/1h/4h, sessiya va katta vaqt oralig'i konteksti, walk-forward (4 blok) bilan baholanadi; push faqat ishonchli ustunlik bo'lgan oraliqlarda yoqiladi.

In [ ]:
if not QUICK:
    !python -m atlas.cli ta-train
    !python -m atlas.cli intraday-train
print(json.dumps({k: {x: v.get(x) for x in ("enabled", "holdout_precision", "holdout_ci90", "holdout_signals", "base_win")}
                  for k, v in json.load(open(f"{DATA}/reports/intraday_report.json"))["timeframes"].items()}, indent=1, default=float))

## 10. Model kartasi (avtomatik)

In [ ]:
from IPython.display import Markdown
wf = rep["walk_forward_summary"]; ab = rep["ablation_tests"]
pre = json.load(open(f"{DATA}/reports/pretrain.json"))
ta = json.load(open(f"{DATA}/reports/ta_report.json"))
best = min(wf, key=lambda v: wf[v]["pinball"])
lines = [f"### ATLAS — model kartasi ({time.strftime('%Y-%m-%d')})",
         f"* Preset: `{PRESET}`, qidiruv sozlamasi ishlatildi: {USE_TUNED}",
         f"* Oldindan o'qitish: val MSE {pre['best_val_mse']:.4f} (o'rtacha bilan to'ldirish {pre['baseline_mean_fill_mse']:.4f}), kesish sanasi {pre['cutoff']}",
         f"* Walk-forward eng yaxshi variant: **{best}** (pinball {wf[best]['pinball']:.4f})",
         *[f"* {k}: farq {v['mean_loss_diff']:+.5f}, p(Holm) = {v['p_holm'] if v['p_holm'] is not None else float('nan'):.3f} → {v['verdict']}" for k, v in ab.items()],
         f"* SetupNet: holdout AUC {ta['auc']:.3f} (tayanch {ta['auc_baseline']:.3f}); ishonchli 30% signal {ta['filter']['top30']['win_rate']:.1f}% vs hammasi {ta['filter']['all']['win_rate']:.1f}%"]
Markdown("\n".join(lines))

## 11. Natijani saytga qaytarish
**A variant — GitHub (avtomatik):** `data` shoxchasiga yangi modellar yoziladi, sayt keyingi yangilanishda ularni ishlatadi.
GitHub → Settings → Developer settings → *Fine-grained token* (faqat `atlas` repo, Contents: Read and write) yarating va so'ralganda kiriting. Token hech qayerda saqlanmaydi.

**B variant — zip:** yuklab oling va kompyuterdagi `ATLAS_AI\data` papkasiga oching.

In [ ]:
PUSH_TO_GITHUB = False  #@param {type:"boolean"}
FILES = ["models/atlas.pt", "models/pretrained.pt", "models/ta_setup.pt", "models/intraday.pt", "models/best_config.json",
         "embeddings/news_cube.npz", "embeddings/reducer.pkl", "reports", "intraday"]
if PUSH_TO_GITHUB:
    from getpass import getpass
    tok = getpass("GitHub token: ")
    url = REPO.replace("https://", f"https://x-access-token:{tok}@")
    !rm -rf /content/push && git clone -q --depth 1 --branch data {url} /content/push
    for f in FILES:
        if os.path.exists(f"{DATA}/{f}"):
            !mkdir -p /content/push/$(dirname {f}) && cp -r {DATA}/{f} /content/push/{os.path.dirname(f) or '.'}/
    !cd /content/push && git config user.name "atlas-colab" && git config user.email "atlas-colab@users.noreply.github.com" && git add -A && git commit -qm "Colab: yangi deep learning modellari" && git push -q origin HEAD:data && echo "Yuborildi"
    del tok, url
!cd {DATA} && zip -qr /content/atlas_models.zip {" ".join(f for f in FILES if os.path.exists(f"{DATA}/{f}"))}
from google.colab import files; files.download("/content/atlas_models.zip")